In [0]:
import pandas as pd

In [0]:
df = spark.read.format("delta").table("`ph-com-ai-us-prod-catalog-2553213839599380`.kerendia_oe_journey.final_mdb_oe_2_0")

In [0]:
!pip install openpyxl

In [0]:
# npi_bhid=spark.read.format('delta').table("`ph-com-ai-us-prod-catalog-2553213839599380`.kerendia_oe_journey.kerendia_oe_npi_bhid_mapping")

In [0]:
# npi_bhid = npi_bhid.select("BH_ID","PROVIDER_ID").distinct()

In [0]:
# npi_bhid = npi_bhid.withColumnRenamed("BH_ID", "CUSTOMER_ID")

In [0]:
# ============================================================================
# EDA Summary — Chunked over features (300 at a time), saved to Delta
# Unique key: (PROVIDER_ID, INSTANCE)
# Serverless-friendly: no cache(), materializes prepped DF to temp Delta path
# Hardened: skips complex Spark types, guards nunique against unhashable cells,
#           and isolates per-column failures so one bad feature can't kill a chunk.
# ============================================================================
import pandas as pd
import numpy as np
from math import ceil
from pyspark.sql.types import (
    DecimalType, DoubleType, ArrayType, MapType, StructType, BinaryType,
)
from pyspark.sql import functions as F

# ── CONFIG ──────────────────────────────────────────────────────────────────
OUTPUT_PATH  = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/oe_journey/oe_volume_space/notebook_output/oe_eda_all"
TEMP_PATH    = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/oe_journey/oe_volume_space/notebook_output/_oe_eda_tmp_input"
Y_COL        = "TOTAL_NBRX_C4"
ID_COLS      = ["PROVIDER_ID", "INSTANCE"]   # composite unique key — kept, not summarized
CHUNK_SIZE   = 300

OUTPUT_COLS = ['feature', 'segment', 'n', 'fill_rate', 'corr_with_y',
               'p25', 'p75', 'mean', 'median', 'n_unique', 'min', 'max', 'std']

COMPLEX_TYPES = (ArrayType, MapType, StructType, BinaryType)

# ── 1. Prep Spark DF in ONE projection (avoids recursion-depth on 18k cols) ─
src = df

assert Y_COL in src.columns, f"Y column '{Y_COL}' not found"
for k in ID_COLS:
    assert k in src.columns, f"ID column '{k}' not found"

# Single select with Decimal→Double cast inline — no loops of withColumn/drop
select_exprs = []
for fld in src.schema.fields:
    if isinstance(fld.dataType, DecimalType):
        select_exprs.append(F.col(fld.name).cast(DoubleType()).alias(fld.name))
    else:
        select_exprs.append(F.col(fld.name))

sdf_prepped = src.select(*select_exprs)

# ── 1b. Materialize to temp Delta (replaces cache on serverless) ───────────
(sdf_prepped.write
   .mode("overwrite")
   .format("delta")
   .option("overwriteSchema", "true")
   .save(TEMP_PATH))

sdf = spark.read.format("delta").load(TEMP_PATH)

n_rows_total = sdf.count()
print(f"Total rows: {n_rows_total:,} | Total cols: {len(sdf.columns):,}")

# Sanity check on uniqueness of composite key
dup_keys = sdf.groupBy(*ID_COLS).count().filter("count > 1").count()
print(f"Duplicate (PROVIDER_ID, INSTANCE) combos: {dup_keys:,}")

# ── 1c. Identify complex-type cols up front — they break .nunique() ────────
complex_cols = {fld.name for fld in sdf.schema.fields
                if isinstance(fld.dataType, COMPLEX_TYPES)}
if complex_cols:
    preview = sorted(complex_cols)[:10]
    print(f"Skipping {len(complex_cols)} complex-type col(s): "
          f"{preview}{'…' if len(complex_cols) > 10 else ''}")

# ── 2. Feature list (exclude Y, IDs, complex types) ────────────────────────
skip_global  = set(ID_COLS) | {Y_COL} | complex_cols
feature_cols = [c for c in sdf.columns if c not in skip_global]
n_features   = len(feature_cols)
n_chunks     = ceil(n_features / CHUNK_SIZE)
print(f"Features to summarize: {n_features:,} | Chunks: {n_chunks}")

# ── 3. Per-feature summary function ────────────────────────────────────────
def _safe_nunique(s: pd.Series) -> float:
    """nunique that survives cells containing lists/arrays (unhashable)."""
    try:
        return s.nunique(dropna=True)
    except TypeError:
        try:
            return s.dropna().map(repr).nunique()
        except Exception:
            return np.nan

def _safe_corr(x: pd.Series, y: pd.Series) -> float:
    try:
        return x.corr(y)
    except Exception:
        return np.nan

def summarize(df_in, label, y_col, y_full, skip_cols):
    rows = []
    n = len(df_in)

    if n == 0:
        return pd.DataFrame(columns=OUTPUT_COLS)

    for col in df_in.columns:
        if col in skip_cols:
            continue

        try:
            s = df_in[col]
            row = {
                'feature':   col,
                'segment':   label,
                'n':         n,
                'fill_rate': s.notna().mean(),
                'n_unique':  _safe_nunique(s),
            }

            if pd.api.types.is_numeric_dtype(s):
                row.update({
                    'p25':    s.quantile(0.25),
                    'p75':    s.quantile(0.75),
                    'mean':   s.mean(),
                    'median': s.median(),
                    'min':    s.min(),
                    'max':    s.max(),
                    'std':    s.std(),
                })

                if (y_full is not None and col != y_col
                        and pd.api.types.is_numeric_dtype(y_full[y_col])
                        and pd.api.types.is_numeric_dtype(y_full[col])):
                    row['corr_with_y'] = _safe_corr(y_full[col], y_full[y_col])
                else:
                    row['corr_with_y'] = np.nan
            else:
                for k in ['p25', 'p75', 'mean', 'median', 'min', 'max', 'std', 'corr_with_y']:
                    row[k] = np.nan

            rows.append(row)

        except Exception as e:
            # Don't let one bad column kill the chunk — log a stub row
            print(f"  ⚠️  Skipped '{col}' ({label}): {type(e).__name__}: {e}")
            stub = {k: np.nan for k in OUTPUT_COLS}
            stub.update({'feature': col, 'segment': label, 'n': n})
            rows.append(stub)

    out = pd.DataFrame(rows)

    # Force numeric dtypes on numeric output columns
    for c in ['fill_rate', 'corr_with_y', 'p25', 'p75', 'mean', 'median',
              'min', 'max', 'std', 'n_unique', 'n']:
        out[c] = pd.to_numeric(out[c], errors='coerce')

    return out[OUTPUT_COLS]


# ── 4. Loop over chunks, summarize, write to Delta ─────────────────────────
for i in range(n_chunks):
    chunk_feats = feature_cols[i * CHUNK_SIZE : (i + 1) * CHUNK_SIZE]
    select_cols = ID_COLS + [Y_COL] + chunk_feats

    print(f"\n── Chunk {i+1}/{n_chunks} — {len(chunk_feats)} features ──")

    pdf_chunk = sdf.select(*select_cols).toPandas()

    # Coerce stringy-numeric object columns within this chunk (skip IDs)
    for c in pdf_chunk.select_dtypes(include='object').columns:
        if c in ID_COLS:
            continue
        try:
            orig_non_null = pdf_chunk[c].notna().sum()
            coerced = pd.to_numeric(pdf_chunk[c], errors='coerce')
            if orig_non_null > 0 and coerced.notna().sum() / orig_non_null >= 0.9:
                pdf_chunk[c] = coerced.astype('float64')
        except Exception as e:
            # Object col may contain unhashable junk — leave as-is, summarize will handle
            print(f"  ⚠️  Coercion skipped for '{c}': {type(e).__name__}: {e}")

    pdf_chunk[Y_COL] = pd.to_numeric(pdf_chunk[Y_COL], errors='coerce')
    y = pdf_chunk[Y_COL]

    skip_cols = set(ID_COLS) | {Y_COL}

    overall_s     = summarize(pdf_chunk,                   'Overall',          Y_COL, pdf_chunk, skip_cols)
    writers_s     = summarize(pdf_chunk[y > 0],            'NBRX > 0',         Y_COL, pdf_chunk, skip_cols)
    non_writers_s = summarize(pdf_chunk[y.fillna(0) == 0], 'NBRX = 0 or NULL', Y_COL, pdf_chunk, skip_cols)

    combined = pd.concat([overall_s, writers_s, non_writers_s], ignore_index=True)
    combined = combined.sort_values(['feature', 'segment']).reset_index(drop=True)

    out_sdf = spark.createDataFrame(combined)

    write_mode = "overwrite" if i == 0 else "append"
    writer = out_sdf.write.mode(write_mode).format("delta")
    if i == 0:
        writer = writer.option("overwriteSchema", "true")
    else:
        writer = writer.option("mergeSchema", "true")
    writer.save(OUTPUT_PATH)

    print(f"  Chunk {i+1} written ({write_mode}) — {len(combined):,} summary rows")

    del pdf_chunk, overall_s, writers_s, non_writers_s, combined, out_sdf

# ── 5. Cleanup temp input ──────────────────────────────────────────────────
try:
    dbutils.fs.rm(TEMP_PATH, recurse=True)
    print(f"\n🧹 Cleaned up temp path: {TEMP_PATH}")
except Exception as e:
    print(f"\n⚠️ Could not clean up {TEMP_PATH}: {e}")

print(f"\n✅ All {n_chunks} chunks written to: {OUTPUT_PATH}")

In [0]:
eda_df = spark.read.format('delta').load("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/oe_journey/oe_volume_space/notebook_output/oe_eda_all")

In [0]:
eda_df.display()